In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib # Para salvar o modelo e o pré-processador

# Carregar os dados (ajustar o caminho se o CSV não estiver na mesma pasta)
# Como você moveu para backend/data/, e o notebook está em backend/notebooks/,
# precisamos "voltar" uma pasta para acessar 'data'.
try:
    df = pd.read_csv('../data/desordens_ficticias.csv')
    print("CSV carregado com sucesso.")
except FileNotFoundError:
    print("Erro: 'desordens_ficticias.csv' não encontrado. Certifique-se de que está na pasta '../data/'.")
    exit() # Sai do script se o arquivo não for encontrado

# Pré-processamento dos dados
# Converter colunas categóricas em numéricas usando LabelEncoder
le_tipo_desordem = LabelEncoder()
df['Tipo_Desordem_encoded'] = le_tipo_desordem.fit_transform(df['Tipo_Desordem'])

le_local_desordem = LabelEncoder()
df['Local_Desordem_encoded'] = le_local_desordem.fit_transform(df['Local_Desordem'])

le_periodo_dia = LabelEncoder()
df['Periodo_Dia_encoded'] = le_periodo_dia.fit_transform(df['Periodo_Dia'])

le_nivel_severidade = LabelEncoder()
df['Nivel_Severidade_encoded'] = le_nivel_severidade.fit_transform(df['Nivel_Severidade'])

# Extrair features de data/hora (ex: dia da semana, hora do dia)
df['Data_Hora'] = pd.to_datetime(df['Data_Hora'])
df['Dia_Semana'] = df['Data_Hora'].dt.dayofweek # 0=Segunda, 6=Domingo
df['Hora_Dia'] = df['Data_Hora'].dt.hour

# Definir features (X) e target (y)
features = [
    'Tipo_Desordem_encoded', 'Local_Desordem_encoded', 'Latitude', 'Longitude',
    'Periodo_Dia_encoded', 'Nivel_Severidade_encoded', 'Dia_Semana', 'Hora_Dia'
]
X = df[features]
y = df['Risco_Previsao'] # A coluna que queremos prever (0 ou 1)

# Dividir os dados em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar o modelo (RandomForestClassifier como exemplo)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Avaliar o modelo (opcional, para verificar a acurácia)
accuracy = model.score(X_test, y_test)
print(f"Acurácia do modelo: {accuracy:.2f}")

# Salvar o modelo treinado e os LabelEncoders (pré-processadores)
# Salvar na pasta 'models' que é um nível acima do 'notebooks'
joblib.dump(model, '../models/modelo_desordem_preditivo.pkl')
joblib.dump(le_tipo_desordem, '../models/le_tipo_desordem.pkl')
joblib.dump(le_local_desordem, '../models/le_local_desordem.pkl')
joblib.dump(le_periodo_dia, '../models/le_periodo_dia.pkl')
joblib.dump(le_nivel_severidade, '../models/le_nivel_severidade.pkl')

print("Modelo e pré-processadores salvos com sucesso na pasta 'models'.")

CSV carregado com sucesso.
Acurácia do modelo: 0.77
Modelo e pré-processadores salvos com sucesso na pasta 'models'.


In [5]:
%pip install scikit-learn joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
